# 1. Imports & Load Raw Data

In this notebook, we preprocess the raw Fragrantica perfume dataset (`fra_perfumes.csv`) and generate separate, L2-normalized 384-dimensional embeddings for four distinct feature spaces:
1. **Description** (`desc_embs.npy`): Semantic representation of the scent notes and narrative.
2. **Main Accords** (`accord_embs.npy`): Semantic representation of the high-level scent profile (e.g., *woody, citrus, warm spicy*).
3. **Gender** (`gender_embss.npy`): 2D orthogonal gender representation (`men`, `women`, `unisex`) projected into 384D space.
4. **Brand** (`brand_embs.npy`): One-hot encoded brand matrix compressed to 384 dimensions via Principal Component Analysis (PCA).

In [ ]:
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

In [ ]:
data = pd.read_csv("fra_perfumes.csv")
data.head()

# 2. Data Cleaning & Feature Engineering

### Bayesian Weighted Rating (`bayesian_avg`)
Raw average ratings can be misleading when a perfume has very few votes:
* **Perfume A**: ⭐ 5.0 (from 2 ratings)
* **Perfume B**: ⭐ 4.6 (from 500 ratings)

To account for rating reliability, we compute a **Bayesian weighted average** (the same formula used by IMDb Top 250):
$$W = \frac{v}{v + m} R + \frac{m}{v + m} C$$
where:
* $R$ = the perfume's average rating (`Rating Value`)
* $v$ = number of votes for the perfume (`Rating Count`)
* $C$ = mean rating across the entire dataset
* $m$ = minimum vote threshold required to be listed (set to the 75th percentile of `Rating Count`)

### Preprocessing Pipeline (`clean_data`)
1. **Name & Brand Extraction**: Strips the gender suffix from `Name`, extracts `Brand` from the last token of `Name`, and removes `Brand` from `Name`.
2. **Missing Value & Index Handling**: Drops rows missing a `Description` and calls `reset_index(drop=True)` so that every row's DataFrame index (`0..N-1`) aligns 1-to-1 with the row position in our `.npy` embedding arrays and FAISS index.
3. **Gender Standardization**: Maps `"for women" -> "women"`, `"for men" -> "men"`, and `"for women and men" -> "unisex"`.
4. **Text & Description Normalization**: Strips list brackets/quotes from `Main Accords` and `Perfumers`, and removes the first sentence of `Description` (which redundantly repeats the perfume's name, brand, and target gender) so the text embedding focuses purely on fragrance notes and character.

In [ ]:
def bayesian_avg(R, v, C, m):
    """
    R = perfume's average rating
    v = number of votes (rating count)
    C = mean rating across all perfumes
    m = threshold, minimum vote count
    """
    return (v / (v + m)) * R + (m / (v + m)) * C

def clean_text(text):
    if not isinstance(text, str):
        return ""
    return text.replace("[", "").replace("]", "").replace("'", "")

def clean_accords(a):
    if isinstance(a, str):
        return a.replace(",", " ").replace("/", " ")
    return ""

def clean_data(df):
    data = df.copy()

    # 1. Handle missing descriptions first and reset index so positional indices match embeddings
    data.dropna(subset=["Description"], inplace=True)
    data.reset_index(drop=True, inplace=True)

    # 2. Feature engineering (Name & Brand)
    data["Name"] = [
        b.replace(a, "").strip()
        for a, b in zip(data["Gender"].astype(str), data["Name"].astype(str))
    ]
    data["Brand"] = data["Name"].apply(lambda x: x.split(" ")[-1])
    data["Name"] = [
        b.replace(a, "").strip()
        for a, b in zip(data["Brand"].astype(str), data["Name"].astype(str))
    ]

    # 3. Handle missing values with compatible dtypes
    data["Description"] = data["Description"].fillna("")
    data["Main Accords"] = data["Main Accords"].fillna("")
    data["Perfumers"] = data["Perfumers"].fillna("")
    data["Gender"] = data["Gender"].fillna("for women and men")
    data["url"] = data["url"].fillna("")

    # 4. Convert numeric rating columns safely
    data["Rating Count"] = (
        data["Rating Count"].astype(str).str.replace(",", "", regex=False).fillna("0").astype(int)
    )
    data["Rating Value"] = data["Rating Value"].fillna(0.0).astype(float)

    # 5. Standardize Gender labels
    gender_map = {
        "for women": "women",
        "for men": "men",
        "for women and men": "unisex",
    }
    data["Gender"] = data["Gender"].replace(gender_map)

    # 6. Clean text columns
    for col in ["Main Accords", "Perfumers"]:
        data[col] = data[col].apply(clean_text)

    data["Main Accords"] = data["Main Accords"].apply(clean_accords)

    # Remove the first sentence from Description (often redundant brand/gender intro)
    data["Description"] = data["Description"].apply(
        lambda x: "".join(x.split(".")[1:]).strip() if "." in x else x.strip()
    )

    # 7. Compute Bayesian weighted rating
    C = data["Rating Value"].mean()
    m = data["Rating Count"].quantile(0.75)
    data["Weighted Rating"] = data.apply(
        lambda x: bayesian_avg(x["Rating Value"], x["Rating Count"], C, m), axis=1
    )

    return data[["Name", "Gender", "Main Accords", "Description", "url", "Weighted Rating", "Brand"]]

In [ ]:
df_clean = clean_data(data)
df_clean.to_pickle("data.pkl")
print(f"Cleaned dataset shape: {df_clean.shape}")
df_clean.head()

# 3. Embedding Generation

### Why `SentenceTransformer` (`all-MiniLM-L6-v2`) instead of raw BERT?
* **Raw BERT** produces token-level embeddings and is not pre-trained for cosine similarity out of the box, making it slower and suboptimal for semantic retrieval.
* **SentenceTransformer (`all-MiniLM-L6-v2`)** uses a Siamese network architecture fine-tuned specifically for sentence-level semantic similarity. It maps texts into a compact **384-dimensional** dense vector space quickly and accurately.

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

## 3.1 Gender Embeddings

We encode `Gender` in a 2D unit circle before zero-padding to 384 dimensions:
* **Men / Male**: $[1.0, 0.0]$
* **Women / Female**: $[0.0, 1.0]$
* **Unisex**: $[1/\sqrt{2}, 1/\sqrt{2}] \approx [0.707, 0.707]$

This geometric design ensures:
* `men` and `women` are orthogonal ($\text{cosine similarity} = 0.0$).
* `unisex` lies at a $45^\circ$ angle between `men` and `women` ($\text{cosine similarity} = 0.707$ with both), so unisex fragrances remain partially compatible with men's or women's queries while exact gender matches score $1.0$.

> **Bug Fix Note**: Previously, `df_clean["Gender"]` contained `"men"` and `"women"`, while `GENDER_VECTORS` only had keys `"male"` and `"female"`, causing both to fall back to `[0, 0, 1]`. Both sets of keys are now explicitly supported.

In [ ]:
UNISEX_VEC = np.array([1 / np.sqrt(2), 1 / np.sqrt(2)], dtype=np.float32)

GENDER_VECTORS = {
    "men":    np.array([1.0, 0.0], dtype=np.float32),
    "male":   np.array([1.0, 0.0], dtype=np.float32),
    "women":  np.array([0.0, 1.0], dtype=np.float32),
    "female": np.array([0.0, 1.0], dtype=np.float32),
    "unisex": UNISEX_VEC,
}

def gender_to_vec(g):
    g = str(g).strip().lower()
    return GENDER_VECTORS.get(g, UNISEX_VEC)

def expand_gender_vec(gv, dim=384):
    """Zero-pad 2D gender vector to 384 dimensions to match MiniLM embedding size."""
    return np.pad(gv, (0, dim - len(gv)), mode="constant").astype(np.float32)

gender_embs = np.vstack([expand_gender_vec(gender_to_vec(g)) for g in df_clean["Gender"]]).astype(np.float32)
print("gender_embs shape:", gender_embs.shape)

## 3.2 Text Embeddings (Description & Main Accords)

We encode `Description` and `Main Accords` separately using `all-MiniLM-L6-v2`. Encoding them separately allows us to control their relative importance during late fusion (`W_DESC = 0.45`, `W_ACCORD = 0.25`).

In [ ]:
def embed_text(df, col, model):
    texts = df[col].fillna("").tolist()
    embeddings = model.encode(texts, show_progress_bar=True)
    return np.asarray(embeddings, dtype=np.float32)

desc_embs = embed_text(df_clean, "Description", model)
accord_embs = embed_text(df_clean, "Main Accords", model)

## 3.3 Brand Embeddings via PCA

We build a binary one-hot brand matrix of shape `(N_perfumes, N_brands)` and apply **PCA** (`n_components=384`) to project the brand indicators into a 384-dimensional space matching the dimensionality of our text embeddings.

In [ ]:
def create_brand_matrix(df):
    brands = sorted(df["Brand"].unique())
    brand_to_idx = {b: i for i, b in enumerate(brands)}

    brand_matrix = np.zeros((len(df), len(brands)), dtype=np.float32)
    for i, brand in enumerate(df["Brand"]):
        brand_matrix[i, brand_to_idx[brand]] = 1.0

    return brand_matrix

def generate_brand_embeddings(df, shape=384):
    brand_matrix = create_brand_matrix(df)
    pca = PCA(n_components=shape)
    brand_embs = pca.fit_transform(brand_matrix)
    return brand_embs.astype(np.float32)

brand_embs = generate_brand_embeddings(df_clean, shape=desc_embs.shape[1])
print("brand_embs shape:", brand_embs.shape)

# 4. L2 Normalization & Saving Embeddings

Before saving, we L2-normalize every embedding matrix row-wise ($\|\mathbf{e}\|_2 = 1$). This puts all four modalities on the exact same unit scale so that weighted linear combination in the recommender respects our chosen weights.

In [ ]:
desc_embs = normalize(desc_embs).astype(np.float32)
accord_embs = normalize(accord_embs).astype(np.float32)
gender_embs = normalize(gender_embs).astype(np.float32)
brand_embs = normalize(brand_embs).astype(np.float32)

np.save("desc_embs.npy", desc_embs)
np.save("accord_embs.npy", accord_embs)
np.save("gender_embss.npy", gender_embs)
np.save("brand_embs.npy", brand_embs)
print("All embeddings normalized and saved successfully.")